# cap10_joint_contract_bc_slice_screen_v1

Purpose: test whether the fold-local cap10 joint target contract is learnable by a conservative behavioral-cloning actor before any PPO or runtime promotion.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- Direct comparison baseline: `cap10_foldlocal_joint_target_contract_v1`.
- Role: `diagnose`.
- Changed layer: `model_selection` / `training_objective` only; no production or runtime promotion.
- Target gate for the eventual model remains excess >= +30pp and path MaxDD <= 15% on fixed split plus walk-forward windows.
- This notebook is only a BC fit gate; passing it allows a later PPO screen, not a model promotion.

External basis:
- Qlib framework and workflow separation: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- Backtest overfit discipline / combinatorial validation motivation: https://doi.org/10.21314/JCF.2016.322
- Deflated Sharpe / multiple-testing caution: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551
- Purged CV / leakage discipline: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678
- Conservative offline-RL motivation before policy improvement: https://arxiv.org/abs/2006.04779

Closest historical success: sparse release and drawdown-control BC screens could fit sparse release states and hand off to PPO diagnostics.

Closest historical failure: previous sparse-release PPO screens improved relative return but let drawdown drift above the 15% target, especially in `wf_2022`.

Why this is not a duplicate: this screen uses the new fold-local joint target contract and evaluates per-fold sparse state slices, including release, downtrend, and drawdown-control states, instead of a pooled date-deduped BC fit.


In [1]:
from __future__ import annotations

import json
import math
import os
import random
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
os.chdir(ROOT)

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.config import load_profile

EXPERIMENT = 'cap10_joint_contract_bc_slice_screen_v1'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_foldlocal_joint_target_contract_v1'
CONTRACT_PATH = SOURCE_DIR / 'joint_target_contract.csv'
SOURCE_SELECTION_PATH = SOURCE_DIR / 'selection.json'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'risk_budget_usage_v1'

OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN_PP = 30.0
TARGET_MAX_DD_PCT = 15.0
GLOBAL_MIN_ACCURACY = 0.70
FOLD_MIN_ACCURACY = 0.70
SLICE_MIN_ACCURACY = 0.60
REQUIRED_SLICES = {
    'release_gate_state': 20,
    'is_downtrend_state': 100,
    'drawdown_control_state': 20,
}
SLICE_COLS = tuple(REQUIRED_SLICES) + ('is_rebound_cluster',)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n')

print('root', ROOT)
print('out_dir', OUT_DIR)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
out_dir /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/cap10_joint_contract_bc_slice_screen_v1


In [2]:
contract = pd.read_csv(CONTRACT_PATH)
contract['datetime'] = pd.to_datetime(contract['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in contract.columns:
        contract[col] = contract[col].fillna(False).astype(bool)

source_selection = json.loads(SOURCE_SELECTION_PATH.read_text())
frame = pd.read_pickle(FRAME_PATH)
profile = load_profile(PROFILE_NAME)

unique_dates = int(contract['datetime'].nunique())
contract_rows = int(len(contract))
folds = sorted(contract['fold'].dropna().astype(str).unique().tolist())
pooled_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=contract, profile=profile)
fold_dataset_rows = []
for fold, fold_targets in contract.groupby('fold'):
    dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile)
    fold_dataset_rows.append({
        'fold': str(fold),
        'contract_rows': int(len(fold_targets)),
        'dataset_samples': int(len(dataset.actions)),
        'start': pd.Timestamp(fold_targets['datetime'].min()).date().isoformat(),
        'end': pd.Timestamp(fold_targets['datetime'].max()).date().isoformat(),
        'release_gate_samples': int(dataset.metadata.get('release_gate_state', pd.Series(False)).fillna(False).astype(bool).sum()),
        'downtrend_samples': int(dataset.metadata.get('is_downtrend_state', pd.Series(False)).fillna(False).astype(bool).sum()),
        'drawdown_control_samples': int(dataset.metadata.get('drawdown_control_state', pd.Series(False)).fillna(False).astype(bool).sum()),
    })

dataset_summary = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'feature_frame': str(FRAME_PATH.relative_to(ROOT)),
    'source_contract': str(CONTRACT_PATH.relative_to(ROOT)),
    'contract_rows': contract_rows,
    'unique_dates': unique_dates,
    'pooled_bc_dataset_samples_after_date_dedup': int(len(pooled_dataset.actions)),
    'date_dedup_gap_rows': int(contract_rows - len(pooled_dataset.actions)),
    'folds': folds,
    'fold_dataset_rows': fold_dataset_rows,
    'source_decision': source_selection.get('decision'),
    'source_bc_screen_allowed_next': source_selection.get('bc_screen_allowed_next'),
}
write_json(OUT_DIR / 'dataset_summary.json', dataset_summary)

pd.DataFrame(fold_dataset_rows).to_csv(OUT_DIR / 'fold_dataset_audit.csv', index=False)
display(pd.DataFrame(fold_dataset_rows))
print(json.dumps(json_safe(dataset_summary), indent=2, allow_nan=False))


,fold,contract_rows,dataset_samples,start,end,release_gate_samples,downtrend_samples,drawdown_control_samples
0,wf_2022_train,505,505,2020-01-02,2021-12-31,40,234,49
1,wf_2023_2024_train,756,755,2020-01-02,2022-12-30,40,477,49


{
  "experiment": "cap10_joint_contract_bc_slice_screen_v1",
  "profile": "risk_budget_usage_v1",
  "feature_frame": "data/soxl_profile_balanced_rule_residual_v22.pkl",
  "source_contract": "runtime/model_selection/card/cap10_foldlocal_joint_target_contract_v1/joint_target_contract.csv",
  "contract_rows": 1261,
  "unique_dates": 756,
  "pooled_bc_dataset_samples_after_date_dedup": 755,
  "date_dedup_gap_rows": 506,
  "folds": [
    "wf_2022_train",
    "wf_2023_2024_train"
  ],
  "fold_dataset_rows": [
    {
      "fold": "wf_2022_train",
      "contract_rows": 505,
      "dataset_samples": 505,
      "start": "2020-01-02",
      "end": "2021-12-31",
      "release_gate_samples": 40,
      "downtrend_samples": 234,
      "drawdown_control_samples": 49
    },
    {
      "fold": "wf_2023_2024_train",
      "contract_rows": 756,
      "dataset_samples": 755,
      "start": "2020-01-02",
      "end": "2022-12-30",
      "release_gate_samples": 40,
      "downtrend_samples": 477,
      "d

The pooled dataset audit is part of the anti-overfit gate. Because fold-local target rows can share the same calendar date, a pooled BC fit would silently drop duplicate dates in the current helper and overstate evidence. The actual screen below trains and evaluates one actor per fold for each weight configuration, then selects by worst-fold and worst-slice behavior.


In [3]:
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
WEIGHT_CONFIGS: list[dict[str, Any]] = [
    {
        'config_name': 'plain',
        'complexity': 0,
        'sample_weight_config': None,
        'description': 'No static sample weighting; tests whether the raw joint contract is already learnable.',
    },
    {
        'config_name': 'release4_downtrend2',
        'complexity': 1,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=4.0,
            downtrend_multiplier=2.0,
            weight_cap=20.0,
        ),
        'description': 'Moderate sparse-state weighting for rebound/release plus downtrend control.',
    },
    {
        'config_name': 'joint8_downtrend2_cb99',
        'complexity': 2,
        'sample_weight_config': StateObjectiveBCSampleWeightConfig(
            rebound_multiplier=8.0,
            downtrend_multiplier=2.0,
            objective_weight_power=0.25,
            class_balanced_beta=0.99,
            weight_cap=20.0,
        ),
        'description': 'Stronger sparse-state and class-balanced weighting for tail action fit.',
    },
]

config_rows = []
for item in WEIGHT_CONFIGS:
    cfg = item['sample_weight_config']
    row = {k: v for k, v in item.items() if k != 'sample_weight_config'}
    row['sample_weight_config'] = None if cfg is None else asdict(cfg)
    config_rows.append(row)

pd.DataFrame(config_rows).to_csv(OUT_DIR / 'bc_weight_configs.csv', index=False)
display(pd.DataFrame(config_rows))


,config_name,complexity,description,sample_weight_config
0,plain,0,No static sample weighting; tests whether the ...,None
1,release4_downtrend2,1,Moderate sparse-state weighting for rebound/re...,"{'rebound_multiplier': 4.0, 'downtrend_multipl..."
2,joint8_downtrend2_cb99,2,Stronger sparse-state and class-balanced weigh...,"{'rebound_multiplier': 8.0, 'downtrend_multipl..."


In [4]:
def _metric_loss(value: float, threshold: float) -> float:
    if value is None or not math.isfinite(float(value)):
        return 1.0
    return max(0.0, float(threshold) - float(value))

def _sample_loss(samples: int, min_samples: int) -> float:
    return 0.0 if int(samples) >= int(min_samples) else (int(min_samples) - int(samples)) / max(int(min_samples), 1)

screen_rows: list[dict[str, Any]] = []
slice_rows: list[dict[str, Any]] = []
fold_rows: list[dict[str, Any]] = []
action_rows: list[dict[str, Any]] = []

for item in WEIGHT_CONFIGS:
    config_name = item['config_name']
    complexity = int(item['complexity'])
    sample_weight_config = item['sample_weight_config']
    per_fold_accuracy = []
    per_fold_samples = []
    per_slice_values: dict[str, list[dict[str, float | int]]] = {name: [] for name in REQUIRED_SLICES}
    config_loss = 0.0

    for fold, fold_targets in contract.groupby('fold'):
        fold = str(fold)
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)
        dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile)
        fit = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=TRAIN_CONFIG,
            sample_weight_config=sample_weight_config,
        )
        fold_eval = evaluate_bc_actor(fit.actor, dataset.obs, dataset.actions)
        fold_accuracy = float(fold_eval['accuracy'])
        fold_samples = int(fold_eval['samples'])
        per_fold_accuracy.append(fold_accuracy)
        per_fold_samples.append(fold_samples)
        fold_pass = fold_accuracy >= FOLD_MIN_ACCURACY
        fold_rows.append({
            'config_name': config_name,
            'complexity': complexity,
            'fold': fold,
            'samples': fold_samples,
            'loss': float(fold_eval['loss']),
            'accuracy': fold_accuracy,
            'fold_gate': bool(fold_pass),
            'min_accuracy': FOLD_MIN_ACCURACY,
            'before_accuracy': float(fit.metrics['before_accuracy']),
            'after_accuracy': float(fit.metrics['after_accuracy']),
            'after_loss': float(fit.metrics['after_loss']),
        })
        config_loss += _metric_loss(fold_accuracy, FOLD_MIN_ACCURACY)

        metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            dataset.obs,
            dataset.actions,
            dataset.metadata,
            slice_cols=SLICE_COLS,
        )
        for row in metrics.to_dict('records'):
            row = dict(row)
            row.update({'config_name': config_name, 'complexity': complexity, 'fold': fold})
            min_samples = REQUIRED_SLICES.get(str(row['slice']), 0)
            row['min_samples'] = int(min_samples)
            row['min_accuracy'] = SLICE_MIN_ACCURACY if str(row['slice']) in REQUIRED_SLICES else None
            row['slice_gate'] = (
                str(row['slice']) not in REQUIRED_SLICES
                or (int(row['samples']) >= int(min_samples) and float(row['accuracy']) >= SLICE_MIN_ACCURACY)
            )
            slice_rows.append(row)
            if str(row['slice']) in REQUIRED_SLICES:
                per_slice_values[str(row['slice'])].append({
                    'samples': int(row['samples']),
                    'accuracy': float(row['accuracy']),
                })
                config_loss += _metric_loss(float(row['accuracy']), SLICE_MIN_ACCURACY)
                config_loss += _sample_loss(int(row['samples']), int(min_samples))

        dist = fit.action_distribution.copy()
        dist['config_name'] = config_name
        dist['complexity'] = complexity
        dist['fold'] = fold
        action_rows.extend(dist.to_dict('records'))

    aggregate: dict[str, Any] = {
        'config_name': config_name,
        'complexity': complexity,
        'description': item['description'],
        'folds': len(per_fold_accuracy),
        'total_samples': int(sum(per_fold_samples)),
        'weighted_accuracy': float(np.average(per_fold_accuracy, weights=per_fold_samples)),
        'min_fold_accuracy': float(min(per_fold_accuracy)),
        'global_accuracy_gate': bool(float(np.average(per_fold_accuracy, weights=per_fold_samples)) >= GLOBAL_MIN_ACCURACY),
        'fold_accuracy_gate': bool(min(per_fold_accuracy) >= FOLD_MIN_ACCURACY),
        'train_steps': int(TRAIN_CONFIG.steps),
        'lr': float(TRAIN_CONFIG.lr),
        'batch_size': int(TRAIN_CONFIG.batch_size),
    }
    slice_gate = True
    for slice_name, min_samples in REQUIRED_SLICES.items():
        values = per_slice_values[slice_name]
        min_acc = min((float(v['accuracy']) for v in values), default=float('nan'))
        min_seen = min((int(v['samples']) for v in values), default=0)
        aggregate[f'{slice_name}_min_accuracy'] = min_acc
        aggregate[f'{slice_name}_min_samples'] = int(min_seen)
        aggregate[f'{slice_name}_gate'] = bool(min_seen >= int(min_samples) and math.isfinite(min_acc) and min_acc >= SLICE_MIN_ACCURACY)
        slice_gate = slice_gate and bool(aggregate[f'{slice_name}_gate'])
    aggregate['slice_gate'] = bool(slice_gate)
    aggregate['bc_screen_gate'] = bool(aggregate['global_accuracy_gate'] and aggregate['fold_accuracy_gate'] and slice_gate)
    aggregate['gate_loss'] = float(config_loss + complexity * 0.001)
    aggregate['ppo_screen_allowed_next'] = bool(aggregate['bc_screen_gate'])
    aggregate['training_allowed_from_this_card'] = False
    aggregate['promotion_allowed'] = False
    screen_rows.append(aggregate)

screen = pd.DataFrame(screen_rows).sort_values(['bc_screen_gate', 'gate_loss', 'complexity'], ascending=[False, True, True]).reset_index(drop=True)
slice_metrics = pd.DataFrame(slice_rows).sort_values(['config_name', 'fold', 'slice']).reset_index(drop=True)
fold_metrics = pd.DataFrame(fold_rows).sort_values(['config_name', 'fold']).reset_index(drop=True)
action_distribution = pd.DataFrame(action_rows).sort_values(['config_name', 'fold', 'action']).reset_index(drop=True)

screen.to_csv(OUT_DIR / 'bc_weight_screen.csv', index=False)
slice_metrics.to_csv(OUT_DIR / 'state_slice_bc_metrics.csv', index=False)
fold_metrics.to_csv(OUT_DIR / 'fold_bc_metrics.csv', index=False)
action_distribution.to_csv(OUT_DIR / 'bc_action_distribution.csv', index=False)

display(screen)
display(slice_metrics[slice_metrics['slice'].isin(['release_gate_state', 'is_downtrend_state', 'drawdown_control_state'])])


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,config_name,complexity,description,folds,total_samples,weighted_accuracy,min_fold_accuracy,global_accuracy_gate,fold_accuracy_gate,train_steps,...,is_downtrend_state_gate,drawdown_control_state_min_accuracy,drawdown_control_state_min_samples,drawdown_control_state_gate,slice_gate,bc_screen_gate,gate_loss,ppo_screen_allowed_next,training_allowed_from_this_card,promotion_allowed
0,plain,0,No static sample weighting; tests whether the ...,2,1260,0.794444,0.765563,True,True,300,...,True,0.979592,49,True,True,True,0.000000,True,False,False
1,release4_downtrend2,1,Moderate sparse-state weighting for rebound/re...,2,1260,0.757937,0.727152,True,True,300,...,True,0.959184,49,True,True,True,0.001000,True,False,False
2,joint8_downtrend2_cb99,2,Stronger sparse-state and class-balanced weigh...,2,1260,0.735714,0.688742,True,False,300,...,True,0.959184,49,True,True,False,0.013258,False,False,False


,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share,config_name,complexity,fold,min_samples,min_accuracy,slice_gate
1,drawdown_control_state,49,0.959184,0.947302,3,3,4,0.857143,4,0.897959,joint8_downtrend2_cb99,2,wf_2022_train,20,0.6,True
2,is_downtrend_state,234,0.863248,1.043702,4,5,1,0.431624,1,0.435897,joint8_downtrend2_cb99,2,wf_2022_train,100,0.6,True
4,release_gate_state,40,1.000000,0.906703,1,1,4,1.000000,4,1.000000,joint8_downtrend2_cb99,2,wf_2022_train,20,0.6,True
6,drawdown_control_state,49,0.979592,0.927716,3,3,4,0.857143,4,0.877551,joint8_downtrend2_cb99,2,wf_2023_2024_train,20,0.6,True
7,is_downtrend_state,477,0.779874,1.135192,5,4,1,0.383648,1,0.406709,joint8_downtrend2_cb99,2,wf_2023_2024_train,100,0.6,True
9,release_gate_state,40,0.975000,0.927449,1,2,4,1.000000,4,0.975000,joint8_downtrend2_cb99,2,wf_2023_2024_train,20,0.6,True
11,drawdown_control_state,49,0.979592,0.926025,3,3,4,0.857143,4,0.877551,plain,0,wf_2022_train,20,0.6,True
12,is_downtrend_state,234,0.880342,1.024763,4,4,1,0.431624,1,0.452991,plain,0,wf_2022_train,100,0.6,True
14,release_gate_state,40,0.975000,0.930612,1,2,4,1.000000,4,0.975000,plain,0,wf_2022_train,20,0.6,True
16,drawdown_control_state,49,0.979592,0.930525,3,3,4,0.857143,4,0.877551,plain,0,wf_2023_2024_train,20,0.6,True


In [5]:
passed = screen[screen['bc_screen_gate'].astype(bool)]
if not passed.empty:
    selected = passed.sort_values(['complexity', 'gate_loss']).iloc[0].to_dict()
    decision = 'joint_contract_bc_slice_ready_for_ppo_screen_no_promotion'
    next_mechanism = 'run_walkforward_ppo_screen_with_bc_warmstart_and_drawdown_drift_stop_rules'
else:
    selected = screen.sort_values(['gate_loss', 'complexity']).iloc[0].to_dict()
    decision = 'joint_contract_bc_slice_failed_no_ppo'
    next_mechanism = 'retune_joint_contract_weighting_or_state_head_before_ppo'

selected_config = str(selected['config_name'])
selected_slices = slice_metrics[slice_metrics['config_name'].eq(selected_config)].copy()
selected_folds = fold_metrics[fold_metrics['config_name'].eq(selected_config)].copy()

failed_checks = []
if not bool(selected['global_accuracy_gate']):
    failed_checks.append('global_accuracy')
if not bool(selected['fold_accuracy_gate']):
    failed_checks.append('fold_accuracy')
for name in REQUIRED_SLICES:
    if not bool(selected[f'{name}_gate']):
        failed_checks.append(name)

slice_summary = {
    'experiment': EXPERIMENT,
    'selected_config': selected_config,
    'decision': decision,
    'bc_screen_gate': bool(selected['bc_screen_gate']),
    'failed_checks': failed_checks,
    'required_slices': REQUIRED_SLICES,
    'slice_min_accuracy': SLICE_MIN_ACCURACY,
    'global_min_accuracy': GLOBAL_MIN_ACCURACY,
    'fold_min_accuracy': FOLD_MIN_ACCURACY,
    'selected': selected,
    'selected_slice_rows': selected_slices.to_dict('records'),
    'selected_fold_rows': selected_folds.to_dict('records'),
}
write_json(OUT_DIR / 'bc_slice_summary.json', slice_summary)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'selected_config': selected_config,
    'selected_gate_loss': float(selected['gate_loss']),
    'selected_weighted_accuracy': float(selected['weighted_accuracy']),
    'selected_min_fold_accuracy': float(selected['min_fold_accuracy']),
    'bc_screen_gate': bool(selected['bc_screen_gate']),
    'ppo_screen_allowed_next': bool(selected['bc_screen_gate']),
    'training_allowed_from_this_card': False,
    'promotion_allowed': False,
    'promotion_blocker': 'BC slice fit is not a backtest; promotion still requires fixed-split and wf_2020/wf_2022/wf_2023_2024 replay with excess >= +30pp and MaxDD <= 15%.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN_PP,
    'target_max_dd_pct': TARGET_MAX_DD_PCT,
    'failed_checks': failed_checks,
    'next_mechanism': next_mechanism,
    'source_contract': str(CONTRACT_PATH.relative_to(ROOT)),
    'source_decision': source_selection.get('decision'),
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_foldlocal_joint_target_contract_v1',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'training_objective'],
    'external_sources': source_selection.get('external_sources', []),
    'wf_2020_coverage': False,
    'actor_retention': 'no actor retained; rerun this notebook to regenerate fold-local BC actors',
    'sidecars': [
        'dataset_summary.json',
        'fold_dataset_audit.csv',
        'bc_weight_configs.csv',
        'bc_weight_screen.csv',
        'state_slice_bc_metrics.csv',
        'fold_bc_metrics.csv',
        'bc_action_distribution.csv',
        'bc_slice_summary.json',
        'selection.json',
        'summary.json',
    ],
}
summary = {
    **selection,
    'screen_rows': screen.to_dict('records'),
}
write_json(OUT_DIR / 'selection.json', selection)
write_json(OUT_DIR / 'summary.json', summary)

print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


{
  "experiment": "cap10_joint_contract_bc_slice_screen_v1",
  "created_at": "2026-05-25T23:15:38.660119+00:00",
  "decision": "joint_contract_bc_slice_ready_for_ppo_screen_no_promotion",
  "selected_config": "plain",
  "selected_gate_loss": 0.0,
  "selected_weighted_accuracy": 0.7944444444444444,
  "selected_min_fold_accuracy": 0.7655629139072848,
  "bc_screen_gate": true,
  "ppo_screen_allowed_next": true,
  "training_allowed_from_this_card": false,
  "promotion_allowed": false,
  "promotion_blocker": "BC slice fit is not a backtest; promotion still requires fixed-split and wf_2020/wf_2022/wf_2023_2024 replay with excess >= +30pp and MaxDD <= 15%.",
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "failed_checks": [],
  "next_mechanism": "run_walkforward_ppo_screen_with_bc_warmstart_and_drawdown_drift_stop_rules",
  "source_contract": "runtime/model_selection/card/cap10_foldlocal_joint_target_contract_v1/joint_target_contract.csv",
  "source_decision": "foldlocal_j

In [6]:
selection = json.loads((OUT_DIR / 'selection.json').read_text())
screen = pd.read_csv(OUT_DIR / 'bc_weight_screen.csv')
fold_metrics = pd.read_csv(OUT_DIR / 'fold_bc_metrics.csv')
slice_metrics = pd.read_csv(OUT_DIR / 'state_slice_bc_metrics.csv')
selected_config = selection['selected_config']
selected_row = screen[screen['config_name'].eq(selected_config)].iloc[0]
selected_folds = fold_metrics[fold_metrics['config_name'].eq(selected_config)]
selected_required = slice_metrics[
    slice_metrics['config_name'].eq(selected_config)
    & slice_metrics['slice'].isin(list(REQUIRED_SLICES))
]

required_lines = []
for name in REQUIRED_SLICES:
    rows = selected_required[selected_required['slice'].eq(name)]
    required_lines.append(
        f"- {name}: min accuracy {rows['accuracy'].min():.3f}, min samples {int(rows['samples'].min())}, gate {bool(selected_row[f'{name}_gate'])}"
    )

conclusion = f"""
## Conclusion

Decision: `{selection['decision']}`.

Selected BC config: `{selected_config}` with weighted fold accuracy `{selected_row['weighted_accuracy']:.3f}`, min fold accuracy `{selected_row['min_fold_accuracy']:.3f}`, and gate loss `{selected_row['gate_loss']:.4f}`.

Fold fit:
""" + "\n".join(
    f"- {row.fold}: accuracy {row.accuracy:.3f}, samples {int(row.samples)}, gate {bool(row.fold_gate)}"
    for row in selected_folds.itertuples(index=False)
) + "\n\nRequired sparse-state slice fit:\n" + "\n".join(required_lines) + f"""

Risk and promotion note: this is only a BC learnability screen. `promotion_allowed=false` and `training_allowed_from_this_card=false`; a later PPO screen is allowed only when `bc_screen_gate=true`, and runtime promotion still requires fixed-split plus `wf_2020`, `wf_2022`, and `wf_2023_2024` replay against excess >= +30pp and MaxDD <= 15%.

Next action: `{selection['next_mechanism']}`.

Artifacts: `bc_weight_screen.csv`, `state_slice_bc_metrics.csv`, `fold_bc_metrics.csv`, `bc_slice_summary.json`, `selection.json`, `summary.json`.
"""
display(Markdown(conclusion))



## Conclusion

Decision: `joint_contract_bc_slice_ready_for_ppo_screen_no_promotion`.

Selected BC config: `plain` with weighted fold accuracy `0.794`, min fold accuracy `0.766`, and gate loss `0.0000`.

Fold fit:
- wf_2022_train: accuracy 0.838, samples 505, gate True
- wf_2023_2024_train: accuracy 0.766, samples 755, gate True

Required sparse-state slice fit:
- release_gate_state: min accuracy 0.975, min samples 40, gate True
- is_downtrend_state: min accuracy 0.799, min samples 234, gate True
- drawdown_control_state: min accuracy 0.980, min samples 49, gate True

Risk and promotion note: this is only a BC learnability screen. `promotion_allowed=false` and `training_allowed_from_this_card=false`; a later PPO screen is allowed only when `bc_screen_gate=true`, and runtime promotion still requires fixed-split plus `wf_2020`, `wf_2022`, and `wf_2023_2024` replay against excess >= +30pp and MaxDD <= 15%.

Next action: `run_walkforward_ppo_screen_with_bc_warmstart_and_drawdown_drift_stop_rules`.

Artifacts: `bc_weight_screen.csv`, `state_slice_bc_metrics.csv`, `fold_bc_metrics.csv`, `bc_slice_summary.json`, `selection.json`, `summary.json`.
